In [3]:
!pip install roboflow ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 683.6 kB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 3.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 65.3 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.5/63.5 kB 4.7 MB/s eta 0:00:00


In [4]:
from roboflow import Roboflow

rf = Roboflow(api_key="x19TylcRGNSRhd1GEwbe")

In [5]:
project = rf.workspace("solar-panel-detection-pz2ap").project("crack-solar-panel")
dataset = project.version(1).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Crack-Solar-Panel-1 in yolov8:: 100%|██████████| 1870/1870 [00:00<00:00, 10815.44it/s]


Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [6]:
project2 = rf.workspace("solarvision-gwljt").project("solar-panel-fault-detection")
dataset2 = project2.version(2).download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Solar-Panel-Fault-Detection-2 in yolov8:: 100%|██████████| 1854/1854 [00:00<00:00, 8478.34it/s]


In [7]:
from roboflow import Roboflow

# Dataset 2
project2 = rf.workspace(
    "solarvision-gwljt"
).project(
    "solar-panel-fault-detection"
)

dataset2 = project2.version(2).download("yolov8")

print("✅ Dataset 2 downloaded successfully")
print("Location:", dataset2.location)

loading Roboflow workspace...
loading Roboflow project...
✅ Dataset 2 downloaded successfully
Location: /kaggle/working/Solar-Panel-Fault-Detection-2


In [8]:
from roboflow import Roboflow
from pathlib import Path
import shutil

# ============================================================
# DOWNLOAD DATASET 2 AGAIN
# ============================================================

rf = Roboflow(api_key="x19TylcRGNSRhd1GEwbe")

project2 = rf.workspace(
    "solarvision-gwljt"
).project(
    "solar-panel-fault-detection"
)

print("Downloading Dataset 2...")

dataset2 = project2.version(2).download(
    "yolov8",
    location="/content/solar_fault_dataset"
)

print("\n" + "=" * 60)
print("DOWNLOAD RESULT")
print("=" * 60)

print("Location:", dataset2.location)

# فحص الملفات
dataset_path = Path("/content/solar_fault_dataset")

print("\nDataset contents:")

if dataset_path.exists():
    for item in dataset_path.iterdir():
        print("📁" if item.is_dir() else "📄", item.name)

    yaml_path = dataset_path / "data.yaml"

    print("\nData.yaml exists:", yaml_path.exists())

else:
    print("❌ Dataset directory was not created")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/solar_fault_dataset in yolov8:: 100%|██████████| 1854/1854 [00:00<00:00, 6688.71it/s]


DOWNLOAD RESULT
Location: /content/solar_fault_dataset

Dataset contents:
📁 valid
📄 README.dataset.txt
📁 test
📄 README.roboflow.txt
📄 data.yaml
📁 train

Data.yaml exists: True


In [9]:
import yaml
from pathlib import Path

# ============================================================
# DATASET 2 - CLASS VERIFICATION
# ============================================================

dataset2_path = Path("/content/solar_fault_dataset")
yaml_path = dataset2_path / "data.yaml"

print("=" * 60)
print("DATASET 2 CLASS VERIFICATION")
print("=" * 60)

with open(yaml_path, "r") as f:
    data2 = yaml.safe_load(f)

print("Classes:", data2["names"])
print("Number of classes:", data2["nc"])

print("\n" + "=" * 60)
print("DATASET 2 STRUCTURE")
print("=" * 60)

for folder in ["train", "valid", "test"]:
    folder_path = dataset2_path / folder

    images_path = folder_path / "images"
    labels_path = folder_path / "labels"

    print(f"\n{folder.upper()}:")
    print("  Images:", len(list(images_path.glob("*"))) if images_path.exists() else 0)
    print("  Labels:", len(list(labels_path.glob("*.txt"))) if labels_path.exists() else 0)

DATASET 2 CLASS VERIFICATION
Classes: ['BakimGereken', 'Cracked', 'Dirty', 'Good', 'Saglam']
Number of classes: 5

DATASET 2 STRUCTURE

TRAIN:
  Images: 797
  Labels: 797

VALID:
  Images: 82
  Labels: 82

TEST:
  Images: 42
  Labels: 42


In [10]:
from pathlib import Path
from collections import Counter

dataset2_path = Path("/content/solar_fault_dataset")

print("=" * 60)
print("DATASET 2 LABEL DISTRIBUTION")
print("=" * 60)

class_counts = Counter()

for split in ["train", "valid", "test"]:

    labels_dir = dataset2_path / split / "labels"

    split_counts = Counter()

    for label_file in labels_dir.glob("*.txt"):

        with open(label_file, "r") as f:

            for line in f:
                line = line.strip()

                if not line:
                    continue

                class_id = int(line.split()[0])

                class_counts[class_id] += 1
                split_counts[class_id] += 1

    print(f"\n{split.upper()}:")

    for class_id in sorted(split_counts):
        print(
            f"Class {class_id}: "
            f"{split_counts[class_id]} objects"
        )


print("\n" + "=" * 60)
print("TOTAL")
print("=" * 60)

class_names = [
    "BakimGereken",
    "Cracked",
    "Dirty",
    "Good",
    "Saglam"
]

for class_id in sorted(class_counts):

    print(
        f"Class {class_id} "
        f"({class_names[class_id]}): "
        f"{class_counts[class_id]} objects"
    )

DATASET 2 LABEL DISTRIBUTION

TRAIN:
Class 0: 109 objects
Class 1: 315 objects
Class 2: 234 objects
Class 3: 237 objects
Class 4: 14 objects

VALID:
Class 0: 20 objects
Class 1: 13 objects
Class 2: 38 objects
Class 3: 21 objects
Class 4: 12 objects

TEST:
Class 0: 14 objects
Class 1: 22 objects
Class 3: 17 objects
Class 4: 1 objects

TOTAL
Class 0 (BakimGereken): 143 objects
Class 1 (Cracked): 350 objects
Class 2 (Dirty): 272 objects
Class 3 (Good): 275 objects
Class 4 (Saglam): 27 objects


In [11]:
import os

# Dataset الأول
print("=== Dataset 1: Crack Solar Panel ===")
for folder in os.listdir("Crack-Solar-Panel-1"):
    print(folder)

print("\n=== Dataset 2: Solar Panel Fault Detection ===")
for folder in os.listdir("Solar-Panel-Fault-Detection-2"):
    print(folder)

=== Dataset 1: Crack Solar Panel ===
train
valid
README.roboflow.txt
data.yaml
README.dataset.txt
test

=== Dataset 2: Solar Panel Fault Detection ===
train
valid
README.roboflow.txt
data.yaml
README.dataset.txt
test


In [12]:
import yaml
from pathlib import Path

print("=" * 60)
print("CHECKING DATASET FILES")
print("=" * 60)

# ============================================================
# Dataset 1
# ============================================================

dataset1_yaml = Path("/content/Crack-Solar-Panel-1/data.yaml")

print("\n=== Dataset 1 ===")

if dataset1_yaml.exists():

    with open(dataset1_yaml, "r") as f:
        data1 = yaml.safe_load(f)

    print("Path:", dataset1_yaml)
    print("Classes:", data1.get("names"))
    print("Number of classes:", data1.get("nc"))

else:
    print("❌ Dataset 1 data.yaml NOT FOUND")
    print("Expected:", dataset1_yaml)


# ============================================================
# Dataset 2
# ============================================================

dataset2_yaml = Path("/content/Solar-Panel-Fault-Detection-2/data.yaml")

print("\n=== Dataset 2 ===")

if dataset2_yaml.exists():

    with open(dataset2_yaml, "r") as f:
        data2 = yaml.safe_load(f)

    print("Path:", dataset2_yaml)
    print("Classes:", data2.get("names"))
    print("Number of classes:", data2.get("nc"))

else:
    print("❌ Dataset 2 data.yaml NOT FOUND")
    print("Expected:", dataset2_yaml)


# ============================================================
# Summary
# ============================================================

print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)

if dataset1_yaml.exists():
    print("Dataset 1:", data1["names"])

if dataset2_yaml.exists():
    print("Dataset 2:", data2["names"])

CHECKING DATASET FILES

=== Dataset 1 ===
❌ Dataset 1 data.yaml NOT FOUND
Expected: /content/Crack-Solar-Panel-1/data.yaml

=== Dataset 2 ===
❌ Dataset 2 data.yaml NOT FOUND
Expected: /content/Solar-Panel-Fault-Detection-2/data.yaml

SUMMARY


In [13]:
import os
import shutil

# تحديد الفئات التي نريدها من Dataset 2
# المفتاح = الرقم القديم، القيمة = الاسم الجديد
class_mapping = {
    0: "Physical_Damage",  # BakimGereken
    1: "Crack",            # Cracked
    2: None,               # Dirty - لا نريده
    3: "Good",             # Good
    4: "Good",             # Saglam = Good أيضاً
}

# الفئات النهائية وأرقامها
final_classes = ["Crack", "Good", "Physical_Damage"]

# إنشاء مجلدات الدمج
for split in ["train", "valid", "test"]:
    os.makedirs(f"merged/{split}/images", exist_ok=True)
    os.makedirs(f"merged/{split}/labels", exist_ok=True)

print("✅ تم إنشاء مجلدات الدمج")

for split in ["train", "valid", "test"]:
    # مسار الصور والـ labels
    img_src = f"Crack-Solar-Panel-1/{split}/images"
    lbl_src = f"Crack-Solar-Panel-1/{split}/labels"

    # نسخ الصور
    for img in os.listdir(img_src):
        shutil.copy(f"{img_src}/{img}", f"merged/{split}/images/ds1_{img}")

    # نسخ الـ labels
    for lbl in os.listdir(lbl_src):
        shutil.copy(f"{lbl_src}/{lbl}", f"merged/{split}/labels/ds1_{lbl}")

print("✅ تم نسخ Dataset 1 بنجاح")

# نسخ Dataset 2 مع تحويل الفئات
for split in ["train", "valid", "test"]:
    img_src = f"/content/solar_fault_dataset/{split}/images"
    lbl_src = f"/content/solar_fault_dataset/{split}/labels"

    # نسخ الصور
    for img in os.listdir(img_src):
        shutil.copy(f"{img_src}/{img}", f"merged/{split}/images/ds2_{img}")

    # تحويل الـ labels
    for lbl in os.listdir(lbl_src):
        with open(f"{lbl_src}/{lbl}", "r") as f:
            lines = f.readlines()

        new_lines = []
        for line in lines:
            parts = line.strip().split()
            old_class = int(parts[0])
            new_class_name = class_mapping[old_class]

            # تجاهل Dirty
            if new_class_name is None:
                continue

            # تحويل الاسم للرقم الجديد
            new_class_num = final_classes.index(new_class_name)
            parts[0] = str(new_class_num)
            new_lines.append(" ".join(parts) + "\n")

        # حفظ الـ label الجديد
        if new_lines:
            with open(f"merged/{split}/labels/ds2_{lbl}", "w") as f:
                f.writelines(new_lines)

print("✅ تم نسخ Dataset 2 مع تحويل الفئات بنجاح")

✅ تم إنشاء مجلدات الدمج
✅ تم نسخ Dataset 1 بنجاح
✅ تم نسخ Dataset 2 مع تحويل الفئات بنجاح


In [14]:
from pathlib import Path

# ============================================================
# REMOVE IMAGES WITHOUT LABELS
# ============================================================

merged_path = Path("/kaggle/working/merged")

print("=" * 70)
print("REMOVING IMAGES WITHOUT LABELS")
print("=" * 70)

total_removed = 0

for split in ["train", "valid", "test"]:

    images_path = merged_path / split / "images"
    labels_path = merged_path / split / "labels"

    removed = 0

    for image_file in images_path.iterdir():

        label_file = labels_path / f"{image_file.stem}.txt"

        if not label_file.exists():

            image_file.unlink()

            removed += 1
            total_removed += 1

    print(f"{split.upper()}: Removed {removed} images")

print("\n" + "=" * 70)
print(f"TOTAL REMOVED: {total_removed}")
print("=" * 70)

REMOVING IMAGES WITHOUT LABELS
TRAIN: Removed 234 images
VALID: Removed 38 images
TEST: Removed 0 images

TOTAL REMOVED: 272


In [15]:
import os
from pathlib import Path
from collections import Counter

# ============================================================
# MERGED DATASET VERIFICATION
# ============================================================

merged_path = Path("/kaggle/working/merged")

print("=" * 70)
print("MERGED DATASET VERIFICATION")
print("=" * 70)

total_images = 0
total_labels = 0

for split in ["train", "valid", "test"]:

    images_path = merged_path / split / "images"
    labels_path = merged_path / split / "labels"

    images = list(images_path.iterdir())
    labels = list(labels_path.glob("*.txt"))

    image_names = {x.stem for x in images}
    label_names = {x.stem for x in labels}

    missing_labels = image_names - label_names
    missing_images = label_names - image_names

    print("\n" + "=" * 50)
    print(split.upper())
    print("=" * 50)

    print("Images :", len(images))
    print("Labels :", len(labels))
    print("Images without labels :", len(missing_labels))
    print("Labels without images :", len(missing_images))

    if len(missing_labels) == 0 and len(missing_images) == 0:
        print("✅ Images and labels are correctly matched")
    else:
        print("⚠️ There are mismatches")

    total_images += len(images)
    total_labels += len(labels)

print("\n" + "=" * 70)
print("TOTAL DATASET")
print("=" * 70)

print("Total Images :", total_images)
print("Total Labels :", total_labels)


# ============================================================
# LABEL VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("LABEL VALIDATION")
print("=" * 70)

class_counts = Counter()
empty_labels = 0
invalid_labels = 0

for split in ["train", "valid", "test"]:

    labels_path = merged_path / split / "labels"

    for label_file in labels_path.glob("*.txt"):

        with open(label_file, "r") as f:
            lines = f.readlines()

        if len(lines) == 0:
            empty_labels += 1
            continue

        for line in lines:

            parts = line.strip().split()

            if len(parts) != 5:
                invalid_labels += 1
                continue

            try:
                class_id = int(parts[0])
                x, y, w, h = map(float, parts[1:])

                # يجب أن تكون الـclasses من 0 إلى 2
                if class_id not in [0, 1, 2]:
                    invalid_labels += 1
                    continue

                # YOLO coordinates يجب أن تكون بين 0 و 1
                if not all(0 <= value <= 1 for value in [x, y, w, h]):
                    invalid_labels += 1
                    continue

                # width و height يجب أن تكون أكبر من صفر
                if w <= 0 or h <= 0:
                    invalid_labels += 1
                    continue

                class_counts[class_id] += 1

            except:
                invalid_labels += 1


print("\nClasses found:")

class_names = {
    0: "Crack",
    1: "Good",
    2: "Physical_Damage"
}

for class_id in [0, 1, 2]:

    print(
        f"Class {class_id} ({class_names[class_id]}): "
        f"{class_counts[class_id]} objects"
    )

print("\nEmpty label files   :", empty_labels)
print("Invalid label entries:", invalid_labels)

if empty_labels == 0 and invalid_labels == 0:
    print("\n✅ All label entries passed validation")
else:
    print("\n⚠️ Some label problems were detected")

MERGED DATASET VERIFICATION

TRAIN
Images : 1376
Labels : 1376
Images without labels : 0
Labels without images : 0
✅ Images and labels are correctly matched

VALID
Images : 121
Labels : 121
Images without labels : 0
Labels without images : 0
✅ Images and labels are correctly matched

TEST
Images : 81
Labels : 81
Images without labels : 0
Labels without images : 0
✅ Images and labels are correctly matched

TOTAL DATASET
Total Images : 1578
Total Labels : 1578

LABEL VALIDATION

Classes found:
Class 0 (Crack): 4673 objects
Class 1 (Good): 302 objects
Class 2 (Physical_Damage): 143 objects

Empty label files   : 0
Invalid label entries: 0

✅ All label entries passed validation


In [16]:
import yaml

data = {
    'train': 'merged/train/images',
    'val': 'merged/valid/images',
    'test': 'merged/test/images',
    'nc': 3,
    'names': ['Crack', 'Good', 'Physical_Damage']
}

with open("merged/data.yaml", "w") as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ تم إنشاء data.yaml بنجاح")

✅ تم إنشاء data.yaml بنجاح


In [17]:
import yaml

yaml_path = "/kaggle/working/merged/data.yaml"

data = {
    "path": "/kaggle/working/merged",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": [
        "Crack",
        "Good",
        "Physical_Damage"
    ]
}

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("✅ data.yaml updated successfully")

with open(yaml_path, "r") as f:
    print(f.read())

✅ data.yaml updated successfully
path: /kaggle/working/merged
train: train/images
val: valid/images
test: test/images
nc: 3
names:
- Crack
- Good
- Physical_Damage



In [18]:
# ============================================================
# YOLOv8 - BASELINE TRAINING
# ============================================================

!pip install ultralytics -q

from ultralytics import YOLO
import os
import yaml

# ============================================================
# 1. التأكد من Dataset
# ============================================================

dataset_path = "/kaggle/working/merged"
yaml_path = os.path.join(dataset_path, "data.yaml")

print("=" * 60)
print("DATASET CONFIGURATION")
print("=" * 60)

if not os.path.exists(yaml_path):
    raise FileNotFoundError(
        f"data.yaml not found at: {yaml_path}"
    )

with open(yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("\nDataset YAML:")
print(data_config)

print("\nClasses:")
for i, name in enumerate(data_config["names"]):
    print(f"Class {i}: {name}")

print(f"\nNumber of classes: {data_config['nc']}")

# ============================================================
# 2. تحميل YOLOv8
# ============================================================

model = YOLO("yolov8n.pt")

print("\n✅ YOLOv8 model loaded successfully")

DATASET CONFIGURATION

Dataset YAML:
{'path': '/kaggle/working/merged', 'train': 'train/images', 'val': 'valid/images', 'test': 'test/images', 'nc': 3, 'names': ['Crack', 'Good', 'Physical_Damage']}

Classes:
Class 0: Crack
Class 1: Good
Class 2: Physical_Damage

Number of classes: 3

✅ YOLOv8 model loaded successfully


In [19]:
for split in ["train", "valid", "test"]:
    imgs_dir = f"merged/{split}/images"
    lbls_dir = f"merged/{split}/labels"

    # نأخذ أسماء الصور بدون امتداد آخر نقطة فقط
    images = set([f.rsplit(".", 1)[0] for f in os.listdir(imgs_dir)])
    labels = set([f.rsplit(".", 1)[0] for f in os.listdir(lbls_dir)])

    no_label = images - labels

    for img_name in no_label:
        for ext in [".jpg", ".jpeg", ".png"]:
            img_path = f"{imgs_dir}/{img_name}{ext}"
            if os.path.exists(img_path):
                os.remove(img_path)
                break

    print(f"✅ {split}: تم حذف {len(no_label)} صورة بدون label")

✅ train: تم حذف 0 صورة بدون label
✅ valid: تم حذف 0 صورة بدون label
✅ test: تم حذف 0 صورة بدون label


In [20]:
for split in ["train", "valid", "test"]:
    images = len(os.listdir(f"merged/{split}/images"))
    labels = len(os.listdir(f"merged/{split}/labels"))
    print(f"{split}: {images} صورة, {labels} label")

train: 1376 صورة, 1376 label
valid: 121 صورة, 121 label
test: 81 صورة, 81 label


In [21]:
# ============================================================
# PREPROCESSING - FULL DATASET
# ============================================================

!pip install opencv-python albumentations numpy pandas matplotlib pillow -q

import cv2
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import os
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


# ============================================================
# Dataset Paths
# ============================================================

dataset_path = Path("/kaggle/working/merged")
output_path = Path("/kaggle/working/preprocessed")

splits = ["train", "valid", "test"]


# ============================================================
# 1- CLAHE
# ============================================================

def apply_clahe(image):

    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_clahe = clahe.apply(l)

    lab_clahe = cv2.merge([
        l_clahe,
        a,
        b
    ])

    image_clahe = cv2.cvtColor(
        lab_clahe,
        cv2.COLOR_LAB2RGB
    )

    return image_clahe


# ============================================================
# 2- Gaussian Blur
# ============================================================

def apply_gaussian_blur(image, kernel_size=3):

    blurred = cv2.GaussianBlur(
        image,
        (kernel_size, kernel_size),
        0
    )

    return blurred


# ============================================================
# 3- Letterbox Resize
# ============================================================

def resize_with_padding(image, target_size=416):

    h, w = image.shape[:2]

    scale = target_size / max(h, w)

    new_h = int(h * scale)
    new_w = int(w * scale)

    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_LINEAR
    )

    top = (target_size - new_h) // 2
    bottom = target_size - new_h - top

    left = (target_size - new_w) // 2
    right = target_size - new_w - left

    padded = cv2.copyMakeBorder(
        resized,
        top,
        bottom,
        left,
        right,
        cv2.BORDER_CONSTANT,
        value=(114, 114, 114)
    )

    return padded, scale, left, top


# ============================================================
# 4- Normalization
# ============================================================

def normalize_image(image, method='imagenet'):

    if method == 'simple':

        normalized = (
            image.astype(np.float32) / 255.0
        )

    elif method == 'imagenet':

        image_float = (
            image.astype(np.float32) / 255.0
        )

        imagenet_mean = np.array([
            0.485,
            0.456,
            0.406
        ])

        imagenet_std = np.array([
            0.229,
            0.224,
            0.225
        ])

        normalized = (
            image_float - imagenet_mean
        ) / imagenet_std

    return normalized


# ============================================================
# YOLO Label Transformation
# ============================================================

def transform_yolo_labels(
    label_path,
    output_label_path,
    original_width,
    original_height,
    scale,
    pad_x,
    pad_y,
    target_size=416
):

    if not label_path.exists():
        return

    new_labels = []

    with open(label_path, "r") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            class_id = parts[0]

            x_center = float(parts[1])
            y_center = float(parts[2])
            box_width = float(parts[3])
            box_height = float(parts[4])

            # YOLO normalized coordinates
            x_center_pixel = x_center * original_width
            y_center_pixel = y_center * original_height

            box_width_pixel = box_width * original_width
            box_height_pixel = box_height * original_height

            # Resize
            x_center_pixel *= scale
            y_center_pixel *= scale

            box_width_pixel *= scale
            box_height_pixel *= scale

            # Padding
            x_center_pixel += pad_x
            y_center_pixel += pad_y

            # Convert back to YOLO normalized coordinates
            new_x_center = x_center_pixel / target_size
            new_y_center = y_center_pixel / target_size

            new_width = box_width_pixel / target_size
            new_height = box_height_pixel / target_size

            new_labels.append(
                f"{class_id} "
                f"{new_x_center:.6f} "
                f"{new_y_center:.6f} "
                f"{new_width:.6f} "
                f"{new_height:.6f}\n"
            )

    with open(output_label_path, "w") as f:
        f.writelines(new_labels)


# ============================================================
# Process Entire Dataset
# ============================================================

total_images = 0

for split in splits:

    print("\n" + "=" * 60)
    print(f"PROCESSING: {split.upper()}")
    print("=" * 60)

    input_images = dataset_path / split / "images"
    input_labels = dataset_path / split / "labels"

    output_images = output_path / split / "images"
    output_labels = output_path / split / "labels"

    output_images.mkdir(
        parents=True,
        exist_ok=True
    )

    output_labels.mkdir(
        parents=True,
        exist_ok=True
    )

    image_files = list(input_images.glob("*"))

    processed_count = 0

    for image_path in image_files:

        if image_path.suffix.lower() not in [
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp"
        ]:
            continue

        # ----------------------------------------------------
        # Load Image
        # ----------------------------------------------------

        image = cv2.imread(str(image_path))

        if image is None:
            print(f"⚠️ Could not read: {image_path.name}")
            continue

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        original_height, original_width = image.shape[:2]


        # ----------------------------------------------------
        # 1- CLAHE
        # ----------------------------------------------------

        image_clahe = apply_clahe(image)


        # ----------------------------------------------------
        # 2- Gaussian Blur
        # ----------------------------------------------------

        # النسخة مع Blur
        with_blur = apply_gaussian_blur(
            image_clahe,
            kernel_size=3
        )

        # ----------------------------------------------------
        # 3- Letterbox Resize
        # ----------------------------------------------------

        resized_image, scale, pad_x, pad_y = \
            resize_with_padding(
                with_blur,
                target_size=416
            )


        # ----------------------------------------------------
        # 4- Normalization
        # ----------------------------------------------------

        normalized_image = normalize_image(
            resized_image,
            method='imagenet'
        )


        # ----------------------------------------------------
        # Convert back to uint8 for saving
        # ----------------------------------------------------

        imagenet_mean = np.array([
            0.485,
            0.456,
            0.406
        ])

        imagenet_std = np.array([
            0.229,
            0.224,
            0.225
        ])

        image_to_save = (
            normalized_image * imagenet_std
            + imagenet_mean
        )

        image_to_save = np.clip(
            image_to_save * 255.0,
            0,
            255
        ).astype(np.uint8)


        # RGB → BGR
        image_to_save = cv2.cvtColor(
            image_to_save,
            cv2.COLOR_RGB2BGR
        )


        # ----------------------------------------------------
        # Save Image
        # ----------------------------------------------------

        output_image_path = (
            output_images / image_path.name
        )

        cv2.imwrite(
            str(output_image_path),
            image_to_save
        )


        # ----------------------------------------------------
        # Transform and Save YOLO Labels
        # ----------------------------------------------------

        label_path = (
            input_labels /
            f"{image_path.stem}.txt"
        )

        output_label_path = (
            output_labels /
            f"{image_path.stem}.txt"
        )

        transform_yolo_labels(
            label_path,
            output_label_path,
            original_width,
            original_height,
            scale,
            pad_x,
            pad_y,
            target_size=416
        )


        processed_count += 1
        total_images += 1


    print(
        f"✅ Processed images: {processed_count}"
    )


# ============================================================
# Final Result
# ============================================================

print("\n" + "=" * 60)
print("PREPROCESSING COMPLETED")
print("=" * 60)

print(f"Total processed images: {total_images}")
print(f"Output dataset: {output_path}")


PROCESSING: TRAIN
✅ Processed images: 1376

PROCESSING: VALID
✅ Processed images: 121

PROCESSING: TEST
✅ Processed images: 81

PREPROCESSING COMPLETED
Total processed images: 1578
Output dataset: /kaggle/working/preprocessed


In [22]:
from pathlib import Path
import os

preprocessed_path = Path("/kaggle/working/preprocessed")

print("=== Checking Preprocessed Dataset ===")

if preprocessed_path.exists():
    print("✅ Preprocessed dataset exists")

    for split in ["train", "valid", "test"]:
        images_path = preprocessed_path / split / "images"
        labels_path = preprocessed_path / split / "labels"

        image_count = len(list(images_path.glob("*"))) if images_path.exists() else 0
        label_count = len(list(labels_path.glob("*.txt"))) if labels_path.exists() else 0

        print(f"\n{split.upper()}:")
        print(f"Images: {image_count}")
        print(f"Labels: {label_count}")
else:
    print("❌ Preprocessed dataset does not exist")

=== Checking Preprocessed Dataset ===
✅ Preprocessed dataset exists

TRAIN:
Images: 1376
Labels: 1376

VALID:
Images: 121
Labels: 121

TEST:
Images: 81
Labels: 81


In [23]:
import yaml
from pathlib import Path

# ============================================================
# Create data.yaml for PREPROCESSED dataset
# ============================================================

preprocessed_path = Path("/kaggle/working/preprocessed")

yaml_path = preprocessed_path / "data.yaml"

data = {
    "path": str(preprocessed_path),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 3,
    "names": [
        "Crack",
        "Good",
        "Physical_Damage"
    ]
}

with open(yaml_path, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("✅ Preprocessed data.yaml created successfully")
print(f"📁 Path: {yaml_path}")

print("\n=== data.yaml ===")
with open(yaml_path, "r") as f:
    print(f.read())

✅ Preprocessed data.yaml created successfully
📁 Path: /kaggle/working/preprocessed/data.yaml

=== data.yaml ===
path: /kaggle/working/preprocessed
train: train/images
val: valid/images
test: test/images
nc: 3
names:
- Crack
- Good
- Physical_Damage



In [24]:
from ultralytics import YOLO
from pathlib import Path
import shutil

# ============================================================
# YOLOv8 TRAINING - PREPROCESSED DATASET
# ============================================================

data_yaml = "/kaggle/working/preprocessed/data.yaml"

run_name = "yolov8_preprocessed"

# Training output
project_dir = Path("/kaggle/working/runs")
run_dir = project_dir / run_name
weights_dir = run_dir / "weights"

# External model directory
export_dir = Path("/kaggle/working/model")
export_dir.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("STARTING YOLOv8 TRAINING")
print("=" * 60)

print(f"Dataset    : {data_yaml}")
print(f"Model      : YOLOv8n")
print(f"Epochs     : 50")
print(f"Batch      : 16")
print(f"Image Size : 416")

# ============================================================
# LOAD PRETRAINED YOLOv8n
# ============================================================

model = YOLO("yolov8n.pt")

print("✅ YOLOv8n loaded successfully")

# ============================================================
# TRAIN
# ============================================================

results = model.train(
    data=data_yaml,

    # Model training
    epochs=50,
    imgsz=416,
    batch=16,

    # Pretrained model
    pretrained=True,

    # Validation
    val=True,

    # Early stopping
    patience=10,

    # Data augmentation
    augment=True,
    mosaic=1.0,
    fliplr=0.5,

    # Reproducibility
    seed=0,
    deterministic=True,

    # Saving
    save=True,
    save_period=1,

    # Output
    project=str(project_dir),
    name=run_name,
    exist_ok=True,

    # Logging / plots
    plots=True,
    verbose=True
)

# ============================================================
# TRAINING FINISHED
# ============================================================

print("\n" + "=" * 60)
print("TRAINING FINISHED")
print("=" * 60)

print(f"Run directory: {run_dir}")

# ============================================================
# CHECK TRAINING WEIGHTS
# ============================================================

print("\n=== Checking Training Weights ===")

best_pt = weights_dir / "best.pt"
last_pt = weights_dir / "last.pt"

if best_pt.exists():
    print(f"✅ best.pt found: {best_pt}")
    print(f"   Size: {best_pt.stat().st_size / (1024**2):.2f} MB")
else:
    print("❌ best.pt NOT FOUND")

if last_pt.exists():
    print(f"✅ last.pt found: {last_pt}")
    print(f"   Size: {last_pt.stat().st_size / (1024**2):.2f} MB")
else:
    print("❌ last.pt NOT FOUND")

# ============================================================
# COPY BEST MODEL TO EXTERNAL MODEL DIRECTORY
# ============================================================

if best_pt.exists():

    external_best = export_dir / "best.pt"

    shutil.copy2(
        best_pt,
        external_best
    )

    print("\n" + "=" * 60)
    print("MODEL EXPORTED SUCCESSFULLY")
    print("=" * 60)

    print(f"✅ External model:")
    print(f"   {external_best}")

    print(
        f"   Size: "
        f"{external_best.stat().st_size / (1024**2):.2f} MB"
    )

else:

    print("\n❌ Cannot export model because best.pt was not found.")

# ============================================================
# LIST FINAL MODEL DIRECTORY
# ============================================================

print("\n=== Final Model Directory ===")

if export_dir.exists():

    for file in export_dir.iterdir():
        print(f"✅ {file}")

# ============================================================
# TEST EXPORTED MODEL
# ============================================================

if (export_dir / "best.pt").exists():

    print("\n=== Testing Exported Model ===")

    test_model = YOLO(
        str(export_dir / "best.pt")
    )

    print("✅ best.pt loaded successfully")
    print(f"Classes: {test_model.names}")

else:

    print("❌ Exported model does not exist.")

STARTING YOLOv8 TRAINING
Dataset    : /kaggle/working/preprocessed/data.yaml
Model      : YOLOv8n
Epochs     : 50
Batch      : 16
Image Size : 416
✅ YOLOv8n loaded successfully
Ultralytics 8.4.126 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/preprocessed/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, lin

In [25]:
from pathlib import Path

print("=== Searching for best.pt ===")

best_files = list(
    Path("/kaggle/working").rglob("best.pt")
)

for f in best_files:
    print("✅", f)

if not best_files:
    print("❌ best.pt not found")

=== Searching for best.pt ===
✅ /kaggle/working/model/best.pt
✅ /kaggle/working/runs/yolov8_preprocessed/weights/best.pt


In [27]:
from IPython.display import FileLink

model_path = "/kaggle/working/model/best.pt"

FileLink(model_path)

/kaggle/working/model/best.pt